In [2]:
import pandas as pd
from pathlib import Path

In [14]:
for path in Path("data/csv").glob("*.csv"):
    print(f"Reading {path}...")
    df = pd.read_csv(path)
    

Reading data\csv\case-accessory.csv...
Reading data\csv\case-fan.csv...
Reading data\csv\case.csv...
Reading data\csv\cpu-cooler.csv...
Reading data\csv\cpu.csv...
Reading data\csv\external-hard-drive.csv...
Reading data\csv\fan-controller.csv...
Reading data\csv\headphones.csv...
Reading data\csv\internal-hard-drive.csv...
Reading data\csv\keyboard.csv...
Reading data\csv\memory.csv...
Reading data\csv\monitor.csv...
Reading data\csv\motherboard.csv...
Reading data\csv\mouse.csv...
Reading data\csv\optical-drive.csv...
Reading data\csv\os.csv...
Reading data\csv\power-supply.csv...
Reading data\csv\sound-card.csv...
Reading data\csv\speakers.csv...
Reading data\csv\thermal-paste.csv...
Reading data\csv\ups.csv...
Reading data\csv\video-card.csv...
Reading data\csv\webcam.csv...
Reading data\csv\wired-network-card.csv...
Reading data\csv\wireless-network-card.csv...


In [3]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv('.env')


# 1. Set your API key (Skip this line if using a .env file or system variable)
API_KEY = os.getenv("OPENAI_API_KEY")

# 2. Initialize the chat model (defaults to gpt-4o-mini or latest standard model)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Test the connection
try:
    response = model.invoke("Hello! If you can read this, our LangChain API test is successful.")
    print("--- Connection Successful! ---")
    print(response.content)
except Exception as e:
    print("--- Connection Failed! ---")
    print(f"Error: {e}")

--- Connection Successful! ---
Hello! It looks like your LangChain API test is indeed successful. How can I assist you further?


In [4]:
from src.database import get_db_connection, execute_read_query
from src.tools import get_database_schema

con = get_db_connection()
res = get_database_schema()

In [4]:
res

'Database Schema Layout:\n\nTable: case_accessory\nCreation SQL:\nCREATE TABLE "case_accessory" (\n"name" TEXT,\n  "price" REAL,\n  "type" TEXT,\n  "form_factor" REAL\n)\n----------------------------------------\n\nTable: case_fan\nCreation SQL:\nCREATE TABLE "case_fan" (\n"name" TEXT,\n  "price" REAL,\n  "size" INTEGER,\n  "color" TEXT,\n  "rpm" TEXT,\n  "airflow" TEXT,\n  "noise_level" TEXT,\n  "pwm" INTEGER\n)\n----------------------------------------\n\nTable: case\nCreation SQL:\nCREATE TABLE "case" (\n"name" TEXT,\n  "price" REAL,\n  "type" TEXT,\n  "color" TEXT,\n  "psu" REAL,\n  "side_panel" TEXT,\n  "external_volume" REAL,\n  "internal_35_bays" INTEGER\n)\n----------------------------------------\n\nTable: cpu_cooler\nCreation SQL:\nCREATE TABLE "cpu_cooler" (\n"name" TEXT,\n  "price" REAL,\n  "rpm" TEXT,\n  "noise_level" TEXT,\n  "color" TEXT,\n  "size" REAL\n)\n----------------------------------------\n\nTable: cpu\nCreation SQL:\nCREATE TABLE "cpu" (\n"name" TEXT,\n  "price

In [5]:
import sys
sys.path.append("..")  # so src/ is importable from the notebook

import os
from dotenv import load_dotenv
from src.tools import *
load_dotenv("../.env")

# Verify env is loaded
print("Model:", os.getenv("MODEL_DEPLOYMENT", "gpt-4o-mini"))
print("API Key set:", bool(os.getenv("OPENAI_API_KEY")))


Model: gpt-4o-mini
API Key set: True


In [7]:
user_input = input("Enter your PC requirements: ")

state_valid = {
    "chat_history": [
        {"role": "user", "content": user_input}
    ],
    "user_requirements": {},
    "current_build": None,
    "logs": [],
    "next_step": "",
    "critique_iterations": 0
}


In [8]:
state_valid

{'chat_history': [{'role': 'user',
   'content': 'The requirement is of a gaming laptop with the good gpu and the budget is $600'}],
 'user_requirements': {},
 'current_build': None,
 'logs': [],
 'next_step': '',
 'critique_iterations': 0}

In [9]:
from src.agent import requirement_gathering_node

# ── Test Case 1: Valid, clear requirements ─────────────────────────────────
# state_valid = {
#     "chat_history": [
#         {"role": "user", "content": "I want to build a gaming PC for around $1000. I prefer AMD processors and NVIDIA GPU."}
#     ],
#     "user_requirements": {},
#     "current_build": None,
#     "logs": [],
#     "next_step": "",
#     "critique_iterations": 0
# }

print("=" * 60)
print("TEST 1: Valid Requirements")
print("=" * 60)
result_valid = requirement_gathering_node(state_valid)
print("next_step    :", result_valid["next_step"])
print("requirements :", result_valid["user_requirements"])
print("logs         :", result_valid["logs"])

# ── Test Case 2: Ambiguous — no budget mentioned ───────────────────────────
state_ambiguous = {
    "chat_history": [
        {"role": "user", "content": "I want a PC for gaming."}
    ],
    "user_requirements": {},
    "current_build": None,
    "logs": [],
    "next_step": "",
    "critique_iterations": 0
}

print("\n" + "=" * 60)
print("TEST 2: Ambiguous Requirements (no budget)")
print("=" * 60)
result_ambiguous = requirement_gathering_node(state_ambiguous)
print("next_step         :", result_ambiguous["next_step"])
print("is_ambiguous      :", result_ambiguous["user_requirements"].get("is_ambiguous"))
print("clarification_msg :", result_ambiguous["user_requirements"].get("clarification_message"))
print("logs              :", result_ambiguous["logs"])

# ── Test Case 3: Conflicting — impossible budget ───────────────────────────
state_conflicting = {
    "chat_history": [
        {"role": "user", "content": "I need a PC to run Cyberpunk 2077 at 4K ultra settings, my budget is $300."}
    ],
    "user_requirements": {},
    "current_build": None,
    "logs": [],
    "next_step": "",
    "critique_iterations": 0
}

print("\n" + "=" * 60)
print("TEST 3: Conflicting Requirements ($300 4K gaming)")
print("=" * 60)
result_conflicting = requirement_gathering_node(state_conflicting)
print("next_step         :", result_conflicting["next_step"])
print("is_conflicting    :", result_conflicting["user_requirements"].get("is_conflicting"))
print("clarification_msg :", result_conflicting["user_requirements"].get("clarification_message"))
print("logs              :", result_conflicting["logs"])

# ── Test Case 4: Empty chat history ───────────────────────────────────────
state_empty = {
    "chat_history": [],
    "user_requirements": {},
    "current_build": None,
    "logs": [],
    "next_step": "",
    "critique_iterations": 0
}

print("\n" + "=" * 60)
print("TEST 4: Empty Chat History (error handling)")
print("=" * 60)
result_empty = requirement_gathering_node(state_empty)
print("next_step :", result_empty["next_step"])
print("logs      :", result_empty["logs"])


TEST 1: Valid Requirements

--- [Node: Gathering & Validating Requirements] ---
-> Validation Alert: Ambiguous=False, Conflicting=True
next_step    : respond_to_user
requirements : {'budget': 600.0, 'primary_use': 'Gaming', 'preferences': None, 'is_ambiguous': False, 'is_conflicting': True, 'clarification_message': 'It seems that a budget of $600 may not be sufficient for a gaming laptop with a good GPU. Gaming laptops typically require a higher budget to accommodate the necessary hardware for optimal performance. Could you please confirm if you are open to adjusting your budget or if you have specific games in mind that you want to play?'}
logs         : ['Requirements invalid. Reason: Input is either ambiguous or conflicting. Asking for clarity.']

TEST 2: Ambiguous Requirements (no budget)

--- [Node: Gathering & Validating Requirements] ---
-> Validation Alert: Ambiguous=True, Conflicting=False
next_step         : respond_to_user
is_ambiguous      : True
clarification_msg : Could y

In [1]:
from src.agent import pc_config_agent, DEFAULT_INITIAL_STATE
import copy

def run():
    state = copy.deepcopy(DEFAULT_INITIAL_STATE)

    print("🖥️  Welcome to the PC Builder Agent!")
    user_input = input("You: ")

    while True:
        # Append user message to history
        state["chat_history"].append({"role": "user", "content": user_input})

        # Run the graph
        state = pc_config_agent.invoke(state)

        # Check what happened
        if state["next_step"] == "awaiting_user":
            # Agent asked a clarification — get user's answer and loop again
            user_input = input("You: ")
            continue

        elif state["next_step"] == "end":
            # Agent finished — check if user wants changes
            follow_up = input("\nYou (feedback or 'exit'): ")
            if follow_up.lower() == "exit":
                print("Goodbye!")
                break
            else:
                user_input = follow_up
                continue



In [2]:
run()

🖥️  Welcome to the PC Builder Agent!

--- [Node: Gathering & Validating Requirements] ---
-> Requirements successfully validated.

--- [Node: Querying Components Database] ---


KeyError: 'respond_to_user'